In [1]:
!nvidia-smi

Wed May 29 21:15:45 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    Off |   00000000:13:00.0 Off |                    0 |
| N/A   28C    P0             N/A /  350W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
from utils import BenchmarkTimer, calc_recall, load_dataset
import torch

In [3]:
WORK_FOLDER = os.path.join("./raft_example")
f = load_dataset("http://ann-benchmarks.com/sift-128-euclidean.hdf5", work_folder=WORK_FOLDER)

metric = f.attrs['distance']
dataset = torch.tensor(f['train'])
queries = torch.tensor(f['test'])
gt_neighbors = torch.tensor(f['neighbors'])
gt_distances = torch.tensor(f['distances'])

The index and data will be saved in ./raft_example


/tmp/ipykernel_3277690/302356110.py:5: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:274.)
  dataset = torch.tensor(f['train'])


In [4]:
print("Dataset shape: ", dataset.shape)
print("Dataset dtype: ", dataset.dtype)
dataset_size = dataset.numel() * dataset.element_size() / 1e9
print("Dataset size in GB: ", dataset_size)
print("Dataset on GPU?: ", dataset.is_cuda)
print("Dataset quantized?: ", dataset.is_quantized)
print("Test query shape: ", queries.shape)
print("Metric for the dataset: ", metric)

Dataset shape:  torch.Size([1000000, 128])
Dataset dtype:  torch.float32
Dataset size in GB:  0.512
Dataset on GPU?:  False
Dataset quantized?:  False
Test query shape:  torch.Size([10000, 128])
Metric for the dataset:  euclidean


In [5]:
dataset_gpu = dataset.cuda()
print("Dataset on GPU?: ", dataset_gpu.is_cuda)
queries_gpu = queries.cuda() 
print("Queries on GPU?: ", dataset_gpu.is_cuda)

Dataset on GPU?:  True
Queries on GPU?:  True


In [15]:
import numpy as np
from pylibraft.neighbors import ivf_flat
from pylibraft.common import DeviceResources

In [16]:
handle = DeviceResources()

In [19]:
%%time
build_params = ivf_flat.IndexParams(
    n_lists=1024,
    metric=metric,
    kmeans_trainset_fraction=0.1,
    kmeans_n_iters=20,
    add_data_on_build=True,
)

index = ivf_flat.build(build_params, dataset_gpu)

CPU times: user 132 ms, sys: 36 ms, total: 168 ms
Wall time: 167 ms


In [20]:
print(index)

Index(type=IVF-FLAT, metric=euclidean, size=1000000, dim=128, n_lists=1024, adaptive_centers=False)


In [25]:
%%time
n_queries=10000
# n_probes is the number of clusters we select in the first (coarse) search step. This is the only hyper parameter for search.
search_params = ivf_flat.SearchParams(n_probes=30)

# Search 10 nearest neighbors.
distances, neighbors = ivf_flat.search(search_params, index, queries_gpu[:n_queries,:], k=10, handle=handle)
handle.sync()

CPU times: user 158 ms, sys: 26 µs, total: 158 ms
Wall time: 155 ms


In [23]:
calc_recall(neighbors, gt_neighbors)

0.97297

In [24]:
n_probes = np.asarray([10, 20, 30, 50, 100, 200, 500, 1024]);
qps = np.zeros(n_probes.shape);
recall = np.zeros(n_probes.shape);

for i in range(len(n_probes)):
    print("\nBenchmarking search with n_probes =", n_probes[i])
    timer = BenchmarkTimer(reps=1, warmup=1)
    for rep in timer.benchmark_runs():
        distances, neighbors = ivf_flat.search(
            ivf_flat.SearchParams(n_probes=n_probes[i]),
            index,
            queries_gpu,
            k=10,
            handle=handle
        )
        handle.sync()
    
    recall[i] = calc_recall(neighbors, gt_neighbors)
    print("recall", recall[i])

    timings = np.asarray(timer.timings)
    avg_time = timings.mean()
    std_time = timings.std()
    qps[i] = queries.shape[0] / avg_time
    print("Average search time: {0:7.3f} +/- {1:7.3} s".format(avg_time, std_time))
    print("Queries per second (QPS): {0:8.0f}".format(qps[i]))


Benchmarking search with n_probes = 10
recall 0.86416
Average search time:   0.054 +/- 0.000128 s
Queries per second (QPS):   185666

Benchmarking search with n_probes = 20
recall 0.94591
Average search time:   0.104 +/- 0.000352 s
Queries per second (QPS):    95971

Benchmarking search with n_probes = 30
recall 0.97297
Average search time:   0.155 +/- 0.000311 s
Queries per second (QPS):    64703

Benchmarking search with n_probes = 50
recall 0.99038
Average search time:   0.254 +/- 0.000152 s
Queries per second (QPS):    39340

Benchmarking search with n_probes = 100
recall 0.99833
Average search time:   0.500 +/- 5.92e-05 s
Queries per second (QPS):    19997

Benchmarking search with n_probes = 200
recall 0.99921
Average search time:   0.984 +/- 0.000186 s
Queries per second (QPS):    10164

Benchmarking search with n_probes = 500
recall 0.99927
Average search time:   1.045 +/-  0.0099 s
Queries per second (QPS):     9566

Benchmarking search with n_probes = 1024
recall 0.99926
Ave

In [27]:
# Save index to file system. Currently only pylibraft supports save/load
ivf_flat.save("sift_ivf_flat.bin", index, handle=handle)

In [23]:
# Determine the max and min value of dataset 
min_dataset_point = dataset_gpu.min().item()
max_dataset_point = dataset_gpu.max().item()

# Assuming uint8 quantization, the range of value after quantization is [0, 255]
quant_min, quant_max = 0, 255 

scale = (max_dataset_point - min_dataset_point) / (quant_max - quant_min) 
# To understand the math here 
zero_point = quant_min - min_dataset_point / scale
zero_point = round(zero_point)
zero_point = int(max(quant_min, min(quant_max, zero_point)))

print("Scale: ", scale, " zero point: ", zero_point)

Scale:  0.8549019607843137  zero point:  0


In [80]:
dataset_gpu_quant = torch.quantize_per_tensor(dataset_gpu, scale=scale, zero_point=zero_point, dtype=torch.quint8)
print("Dataset on GPU?: ", dataset_gpu_quant.is_cuda)
print("Dataset quantized?: ", dataset_gpu_quant.is_quantized)
print(dataset_gpu_quant)

Dataset on GPU?:  True
Dataset quantized?:  True
tensor([[  0.0000,  16.2431,  35.0510,  ...,  24.7922,  23.0824,   0.8549],
        [ 13.6784,  35.0510,  18.8078,  ...,  11.1137,  21.3725,  33.3412],
        [  0.0000,   0.8549,   5.1294,  ...,   4.2745,  23.0824,  10.2588],
        ...,
        [ 29.9216,  11.9686,  11.9686,  ...,  49.5843,  10.2588,   0.0000],
        [  0.0000,   5.1294,  11.9686,  ...,   0.8549,   1.7098,  12.8235],
        [113.7020,  30.7765,   0.0000,  ...,  24.7922,  16.2431,   0.0000]],
       device='cuda:0', size=(1000000, 128), dtype=torch.quint8,
       quantization_scheme=torch.per_tensor_affine, scale=0.8549019607843137,
       zero_point=0)


In [87]:
%%time
index_quant = ivf_flat.build(build_params, torch.int_repr(dataset_gpu_quant))
print(torch.int_repr(dataset_gpu_quant))

tensor([[  0,  19,  41,  ...,  29,  27,   1],
        [ 16,  41,  22,  ...,  13,  25,  39],
        [  0,   1,   6,  ...,   5,  27,  12],
        ...,
        [ 35,  14,  14,  ...,  58,  12,   0],
        [  0,   6,  14,  ...,   1,   2,  15],
        [133,  36,   0,  ...,  29,  19,   0]], device='cuda:0',
       dtype=torch.uint8)
CPU times: user 139 ms, sys: 64.4 ms, total: 204 ms
Wall time: 201 ms


In [89]:
n_probes = np.asarray([10, 20, 30, 50, 100, 200, 500, 1024]);
qps = np.zeros(n_probes.shape);
recall = np.zeros(n_probes.shape);

for i in range(len(n_probes)):
    print("\nBenchmarking search with n_probes =", n_probes[i])
    timer = BenchmarkTimer(reps=1, warmup=1)
    for rep in timer.benchmark_runs():
        distances, neighbors = ivf_flat.search(
            ivf_flat.SearchParams(n_probes=n_probes[i]),
            index_quant,
            queries_gpu,
            k=10
        )
    
    recall[i] = calc_recall(neighbors, gt_neighbors)
    print("recall", recall[i])

    timings = np.asarray(timer.timings)
    avg_time = timings.mean()
    std_time = timings.std()
    qps[i] = queries.shape[0] / avg_time
    print("Average search time: {0:7.3f} +/- {1:7.3} s".format(avg_time, std_time))
    print("Queries per second (QPS): {0:8.0f}".format(qps[i]))


Benchmarking search with n_probes = 10


NameError: name 'uint8' is not defined

In [28]:
from pylibraft.neighbors import cagra

In [29]:
%%time
params = cagra.IndexParams(intermediate_graph_degree=32, graph_degree=16)
cagra_index = cagra.build(params, dataset_gpu)

CPU times: user 29.8 s, sys: 5.28 s, total: 35.1 s
Wall time: 32.8 s


In [30]:
itopk = np.asarray([16, 32, 48, 64, 80, 96, 112, 128]);
qps = np.zeros(n_probes.shape);
recall = np.zeros(n_probes.shape);

for i in range(len(itopk)):
    print("\nBenchmarking search with itopk =", itopk[i])
    timer = BenchmarkTimer(reps=1, warmup=1)
    for rep in timer.benchmark_runs():
        distances, neighbors = cagra.search(
            cagra.SearchParams(itopk_size=itopk[i]),
            cagra_index,
            queries_gpu,
            k=10
        )
    
    recall[i] = calc_recall(neighbors, gt_neighbors)
    print("recall", recall[i])

    timings = np.asarray(timer.timings)
    avg_time = timings.mean()
    std_time = timings.std()
    qps[i] = queries.shape[0] / avg_time
    print("Average search time: {0:7.3f} +/- {1:7.3} s".format(avg_time, std_time))
    print("Queries per second (QPS): {0:8.0f}".format(qps[i]))


Benchmarking search with itopk = 16
recall 0.56865
Average search time:   0.003 +/- 0.000424 s
Queries per second (QPS):  3587635

Benchmarking search with itopk = 32
recall 0.80153
Average search time:   0.003 +/- 0.000301 s
Queries per second (QPS):  3294042

Benchmarking search with itopk = 48
recall 0.88321
Average search time:   0.004 +/- 0.000327 s
Queries per second (QPS):  2342750

Benchmarking search with itopk = 64
recall 0.92246
Average search time:   0.005 +/- 0.000283 s
Queries per second (QPS):  1847820

Benchmarking search with itopk = 80
recall 0.94752
Average search time:   0.007 +/- 0.000324 s
Queries per second (QPS):  1466232

Benchmarking search with itopk = 96
recall 0.95827
Average search time:   0.008 +/- 0.000295 s
Queries per second (QPS):  1304375

Benchmarking search with itopk = 112
recall 0.96987
Average search time:   0.009 +/- 0.000315 s
Queries per second (QPS):  1096397

Benchmarking search with itopk = 128
recall 0.97392
Average search time:   0.010 

In [104]:
%%time
params = cagra.IndexParams(intermediate_graph_degree=32, graph_degree=16)
cagra_index_quant = cagra.build(params, torch.int_repr(dataset_gpu_quant))

CPU times: user 30.1 s, sys: 4.38 s, total: 34.5 s
Wall time: 32.1 s


In [107]:
print(cagra_index_quant)

Index(type=CAGRA, metric=sqeuclidean, metric=0, dim=128, graph_degree=16)


In [31]:
# Save CAGRA graph and convert its index 

In [103]:
itopk = np.asarray([16, 32, 48, 64, 80, 96, 112, 128]);
qps = np.zeros(n_probes.shape);
recall = np.zeros(n_probes.shape);

for i in range(len(itopk)):
    print("\nBenchmarking search with itopk =", itopk[i])
    timer = BenchmarkTimer(reps=1, warmup=1)
    for rep in timer.benchmark_runs():
        distances, neighbors = cagra.search(
            cagra.SearchParams(itopk_size=itopk[i]),
            cagra_index_quant,
            queries_gpu,
            k=10
        )
    
    recall[i] = calc_recall(neighbors, gt_neighbors)
    print("recall", recall[i])

    timings = np.asarray(timer.timings)
    avg_time = timings.mean()
    std_time = timings.std()
    qps[i] = queries.shape[0] / avg_time
    print("Average search time: {0:7.3f} +/- {1:7.3} s".format(avg_time, std_time))
    print("Queries per second (QPS): {0:8.0f}".format(qps[i]))


Benchmarking search with itopk = 16


TypeError: Cannot convert pylibraft.neighbors.cagra.cagra.IndexUint8 to pylibraft.neighbors.cagra.cagra.IndexFloat